# Creating an OME-Zarr dataset

This notebook is based on the online [OME-Zarr textbook](https://ome-zarr-book.readthedocs.io/). There's a lot more material there if you want to dig deeper.

In this notebook, we will work through creating an OME-Zarr dataset from scratch. Don't be intimidated by the number of steps here! The point is to see how the metadata is structured in OME-Zarr and how to manipulate it; we'll look at simpler ways to convert an existing dataset to OME-Zarr in a later notebook.

In [ ]:
import zarr
import imageio.v3 as iio
from rich.pretty import pprint
import matplotlib.pyplot as plt
import math
import scipy.ndimage

## Load an image as a numpy array

Download the `hoa_heart.zarr` dataset from the [ome zarr book repository](https://github.com/ome-zarr-models/ome-zarr-book/tree/main/book/data/hoa_heart.zarr), and put it in the `data/` directory. (The `data` directory should be inside the same folder as this notebook).

This is a downsampled version of a [heart dataset from the Human Organ Atlas](https://doi.esrf.fr/10.15151/ESRF-DC-1773966241).

In [ ]:
heart_np = zarr.open_array("data/hoa_heart.zarr")[:]

In [ ]:
print(heart_np.shape)

In [ ]:
print(heart_np.dtype)

Let's display a slice through it with matplotlib:

In [ ]:
plt.imshow(heart_np[:, :, 60], cmap="gray")

## Converting to Zarr

We can convert this numpy array to Zarr in one step with `zarr-python`:

In [ ]:
zarr.save("data/heart-test.zarr", heart_np)

But let's take a more manual approach so we can configure exactly how the Zarr array is setup:

In [ ]:
# Create an empty zarr array at data/exported-heart.zarr with:
# - chunks of size (32 x 32 x 32)
# - overall size + data type matching our heart numpy array
# - compressed with blosc-zstd at level 3

compressors = zarr.codecs.BloscCodec(cname='zstd', clevel=3)
heart_zarr = zarr.create_array(
    store="data/exported-heart.zarr",
    shape=heart_np.shape,
    chunks=(32, 32, 32),
    dtype=heart_np.dtype,
    compressors=compressors,
    dimension_names=["x", "y", "z"]
)

This creates an empty directory with a `zarr.json` file. If you open that file, you should see the metadata given there matches the options we specified above. For example, a chunk shape of (32, 32, 32).

If we try to plot a slice through our zarr array, we see it is empty:

In [ ]:
plt.imshow(heart_zarr[:, :, 60], cmap="gray")

We have to copy our image data into the array:

In [ ]:
# Copy the heart data into the zarr array
heart_zarr[:] = heart_np

When this is run, you should see a `/c` directory gets created, along with all of the chunks inside.

In [ ]:
# Now we see our data in the zarr array
plt.imshow(heart_zarr[:, :, 60], cmap="gray")

## Convert to OME-Zarr

Let's use `ome-zarr-models` to create an OME-Zarr image instead:

In [ ]:
from ome_zarr_models.v05 import Image
from ome_zarr_models.v05.axes import Axis
from pydantic_zarr.v3 import ArraySpec

voxel_size = 19.89
heart_ome_zarr = Image.new(
    array_specs = [ArraySpec.from_zarr(heart_zarr)],
    paths = ["level0"],
    axes = [
        Axis(name="x", type="space", unit="um"),
        Axis(name="y", type="space", unit="um"),
        Axis(name="z", type="space", unit="um")
    ],
    global_scale = [voxel_size, voxel_size, voxel_size],
    scales = [[1, 1, 1]],
    translations = [[0, 0, 0]],
    name = "heart_image"
)
pprint(heart_ome_zarr)

In [ ]:
# Write the empty OME-Zarr to file
ome_zarr_group = heart_ome_zarr.to_zarr("data/exported-heart.ome.zarr", path="")
print(ome_zarr_group)

You should see a new directory at `data/exported-heart.ome.zarr`. This is a Zarr group, containing a single array 'level0'. You should see that all the OME-Zarr metadata we specified above appears in the top level `zarr.json` inside `attributes`.

In [ ]:
# Open the level 0 array
level0_array = ome_zarr_group["level0"]
print(level0_array.shape)

In [ ]:
# It's empty, as before
plt.imshow(level0_array[:, :, 60], cmap="gray")

In [ ]:
# Let's fill it with our image data
level0_array[:] = heart_zarr[:]
plt.imshow(level0_array[:, :, 60], cmap="gray")

In [ ]:
from pydantic_zarr.v3 import ArraySpec
pprint(ArraySpec.from_zarr(heart_zarr))

## Multi-scale OME-Zarr

So far we've made an OME-Zarr image with one dataset (the full resolution image). How can we store multiple resolution levels in the same OME-Zarr?

Here's our full-res array spec:

In [ ]:
full_res_spec = ArraySpec.from_zarr(heart_zarr)
pprint(full_res_spec)

We need to create new ones for our lower resolution levels. Let's add one more level for now that is 2x smaller:

In [ ]:
downsampled_spec = full_res_spec.model_copy(
    update={"shape": tuple(math.ceil(i / 2) for i in full_res_spec.shape)
})
pprint(downsampled_spec)

Now we update our OME Zarr image:

In [ ]:
multiscale_image = Image.new(
    array_specs = [full_res_spec, downsampled_spec],
    paths = ["level0", "level1"],
    axes = [
        Axis(name="x", type="space", unit="um"),
        Axis(name="y", type="space", unit="um"),
        Axis(name="z", type="space", unit="um")
    ],
    global_scale = [voxel_size, voxel_size, voxel_size],
    scales = [[1, 1, 1], [2, 2, 2]],
    translations = [[0, 0, 0], [0, 0, 0]],
    name = "heart_image"
)
pprint(multiscale_image)

In [ ]:
# Write the multi-scale image to file
multiscale_group = multiscale_image.to_zarr("data/exported-heart-multiscale.ome.zarr", path="")
print(multiscale_group)

You should see that a new directory has been created at `data/exported-heart-multiscale.ome.zarr`. It is a zarr group containing two arrays: level0 and level1. Level0 is our full resolution dataset, and level1 is our 2x downsampled one.

As before, we need to fill them with our imaging data:

In [ ]:
multiscale_group['level0'][:] = heart_zarr[:]
print("full scale image is", heart_zarr.shape)

downsampled_data = scipy.ndimage.zoom(heart_zarr[:], zoom=[0.5, 0.5, 0.5])
print("2x downsampled image is", downsampled_data.shape)
    
multiscale_group['level1'][:] = downsampled_data

To check it worked, let's plot the lowest resolution level:

In [ ]:
level1_array = multiscale_group["level1"]
print(level1_array)

plt.imshow(level1_array[:, :, 30], cmap="gray")

## Exercise

Try expanding the code above to create an OME-Zarr image with four resolution levels:
- the full-reslution dataset
- one  that is 2x smaller than the original
- one that is 4x smaller than the original
- one that is 8x smaller than the original

You may want to update the chunk size at the lowest resolution levels to a smaller size e.g. (16, 16, 16)

You could also try to process one of your own datasets.

## Key take-aways

In summary, we have learnt that:
- Zarr arrays can be created via `zarr-python`, with full control of settings like chunk size, compressor and compresssion level.
- Metadata is written to `zarr.json` files
- OME Zarr adds extra metadata to `attributes` in `zarr.json`
- We can use `ome-zarr-models` to specify OME-Zarr metadata like axes, multiscale dataset sizes...
- Here we have built our OME-Zarr dataset step-by-step, but there are many packages available that can simplify this process